In [167]:
import numpy as np

In [168]:
def softmax(x):
    x = np.array(x)
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)


In [169]:
def gradsoftmax(input):
    l=np.array(input)
    f,g=np.exp(l),np.sum(np.exp(l))
    return (f*g+f**2)/g**2

In [170]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=False)
list(iris)

['data',
 'target',
 'frame',
 'target_names',
 'DESCR',
 'feature_names',
 'filename',
 'data_module']

In [171]:
X=iris.data
y=iris.target

In [172]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)

In [173]:
X_train[0]

array([5. , 3.6, 1.4, 0.2])

In [174]:
len(X_train)

112

In [175]:
W = np.random.randn(4, 3)   
X = np.array(X_train)     
b = np.random.randn(1,3)

logits = X @ W + b         
softmax1 = softmax(logits)


In [176]:
softmax1

array([[2.48694761e-04, 9.78633837e-02, 9.01887922e-01],
       [1.90229482e-04, 1.26528544e-01, 8.73281226e-01],
       [4.54527743e-04, 7.88098831e-01, 2.11446641e-01],
       [2.78280111e-04, 6.96196382e-01, 3.03525338e-01],
       [1.50851887e-04, 4.84196476e-01, 5.15652672e-01],
       [1.27208099e-04, 7.75000174e-02, 9.22372775e-01],
       [1.37983813e-04, 7.68829643e-02, 9.22979052e-01],
       [5.59120653e-04, 3.32277879e-01, 6.67163000e-01],
       [2.73905377e-04, 6.56088905e-01, 3.43637189e-01],
       [2.07517899e-04, 8.03917140e-01, 1.95875343e-01],
       [1.58881196e-04, 3.90251458e-01, 6.09589661e-01],
       [7.06659065e-05, 8.73023635e-01, 1.26905700e-01],
       [4.55298768e-04, 5.62115748e-01, 4.37428954e-01],
       [6.85316524e-05, 7.99044161e-01, 2.00887307e-01],
       [4.91090810e-04, 7.13226243e-01, 2.86282666e-01],
       [2.09492857e-04, 8.09676034e-02, 9.18822904e-01],
       [6.96495144e-05, 7.62770023e-01, 2.37160327e-01],
       [4.08769964e-04, 4.27578

In [177]:
print(iris.target_names[np.argmax(softmax1)%3])

virginica


In [178]:
def cross_entropy(X, y, W,b):
    loss = 0
    for i in range(len(X)):
        logits = X[i] @ W + b           
        probs = softmax(logits)        
        loss += np.sum(y[i] * np.log(probs + 1e-9))
    return -loss / len(X)


In [179]:
cross_entropy(X_train,y_train,W,b)

np.float64(10.288617642061185)

In [180]:
def grad_crossentropy(X,y,W,b):
    grad_w = np.zeros_like(W)
    grad_b = np.zeros_like(b)
    for i in range(len(X)):
        logits = X[i] @ W + b
        probs = softmax(logits)
        grad_w += np.outer(X[i],probs-y[i])
        grad_b += probs-y[i]
    return grad_w / len(X) , grad_b/len(X)


In [181]:
len(X_train)

112

In [182]:
np.random.seed(2)

W = np.random.randn(4, 3)     
b = np.random.randn(1,3)

X_subtrain,X_valid ,y_subtrain,y_valid  = X_train[:90] ,X_train[90:],y_train[:90],y_train[90:]
eta = 0.1
n_epoches = 100
best_valid_error = float('inf')
steps = 0
for epoch in range(n_epoches):
    W -= eta*grad_crossentropy(X_subtrain,y_subtrain,W,b)[0]
    b -= eta*grad_crossentropy(X_subtrain,y_subtrain,W,b)[1]
    val_error = cross_entropy(X_valid,y_valid,W,b)
    steps+=1
    if val_error < best_valid_error:
        best_valid_error = val_error
    else:
        break
 


In [183]:
W,b

(array([[ 0.58684852,  2.43851901,  0.73874503],
        [ 1.921652  , -0.65442207,  0.4795246 ],
        [ 1.72515946,  0.80787401,  1.25605209],
        [-0.4338756 ,  1.27291453,  3.0945044 ]]),
 array([[ 0.2271846 , -0.76640517,  0.9685595 ]]))

In [184]:
X_train[0]
softmax(X_train[0] @ W + b)

array([[0.8560173 , 0.12155234, 0.02243035]])

In [185]:
np.argmax(softmax(X_train[0] @ W + b))

np.int64(0)

In [186]:
y_train[0]

np.int64(0)

In [187]:
score = 0
for i in range(len(X_test)):
    if np.argmax(softmax(X_test[i] @ W + b)) == y_test[i]:
        score +=1

print(score/len(X_test))

0.868421052631579


what about trying to feed my model the all training data

In [188]:
np.random.seed(2)
W = np.random.randn(4, 3)     
b = np.random.randn(1,3)

eta = 0.1
n_epoches = 300
for epoch in range(steps):
    W -= eta*grad_crossentropy(X_train,y_train,W,b)[0]
    b -= eta*grad_crossentropy(X_train,y_train,W,b)[1]


In [189]:
score = 0
for i in range(len(X_test)):
    if np.argmax(softmax(X_test[i] @ W + b)) == y_test[i]:
        score +=1

print(score/len(X_test))

0.868421052631579
